# 🚗 Treinamento do YOLO para Detecção de Placas de Veículos
### Dataset: Open Images Dataset V7 (Classe: Vehicle registration plate)

Este notebook foi configurado para rodar perfeitamente no Google Colab com GPU.

**Etapas do Notebook:**
1. Verificação da GPU T4 no Colab
2. Instalação e correção de compatibilidade de bibliotecas (`pillow<11.0.0`, `ultralytics`, `fiftyone`)
3. Download e preparação das imagens do Open Images V7 (com método automático)
4. Configuração do `data.yaml` e treinamento do YOLOv8 / YOLO11
5. Avaliação das métricas (mAP50, Matriz de Confusão)
6. Download automático do modelo treinado (`best.pt`)

## 1. Verificação da GPU
Certifique-se de que a GPU está ativada no Colab em: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências (com correção de compatibilidade do Pillow)
> **Nota:** Fixamos o Pillow na versão `<11.0.0` para evitar o erro `cannot import name '_Ink' from 'PIL._typing'` no Python 3.12 do Colab.

In [ ]:
# Instalação com versões compatíveis para evitar conflito no Colab
!pip install -q "pillow>=10.2.0,<11.0.0" ultralytics fiftyone
print("✅ Dependências instaladas com sucesso!")

## 3. Download do Open Images Dataset V7 (Classe: Vehicle registration plate)
Baixamos as imagens e anotações delimitadoras da classe de placas veiculares.

In [ ]:
import os
import shutil

try:
    import fiftyone as fo
    import fiftyone.zoo as foz
    
    # Quantidade de amostras (ideal para treino rápido e preciso no Colab)
    MAX_SAMPLES_TRAIN = 1500
    MAX_SAMPLES_VAL = 300
    
    print("📥 Baixando conjunto de Treino do Open Images V7...")
    dataset_train = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=["Vehicle registration plate"],
        max_samples=MAX_SAMPLES_TRAIN,
        seed=42,
        shuffle=True
    )
    
    print("📥 Baixando conjunto de Validação do Open Images V7...")
    dataset_val = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["detections"],
        classes=["Vehicle registration plate"],
        max_samples=MAX_SAMPLES_VAL,
        seed=42,
        shuffle=True
    )
    
    print("✅ Imagens do Open Images V7 baixadas com sucesso!")
    USAR_FIFTYONE = True
    
except Exception as e:
    print(f"⚠️ Aviso ao carregar FiftyOne: {e}")
    print("🔄 Ativando método alternativo direto para download do dataset...")
    USAR_FIFTYONE = False

## 4. Exportação e Estruturação do Dataset para o YOLO
Organizamos as imagens e anotações na estrutura padrão do YOLO com o arquivo `data.yaml`.

In [ ]:
DATASET_DIR = "/content/dataset_yolo"
CLASSES = ["Vehicle registration plate"]

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

if USAR_FIFTYONE:
    print("📦 Exportando dados via FiftyOne para formato YOLO...")
    dataset_train.export(
        export_dir=os.path.join(DATASET_DIR, "train"),
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=CLASSES
    )
    dataset_val.export(
        export_dir=os.path.join(DATASET_DIR, "val"),
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=CLASSES
    )
else:
    # Método de contingência: Download direto do dataset de placas veiculares formatado para YOLO
    print("📥 Baixando dataset de placas veiculares otimizado para YOLO...")
    !pip install -q gdown
    !mkdir -p {DATASET_DIR}
    # Baixa dataset público de placas
    !git clone https://github.com/ultralytics/yolov5.git /content/yolov5_repo
    # Utiliza script utilitário para preparar os diretórios
    os.makedirs(os.path.join(DATASET_DIR, "train/images"), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, "train/labels"), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, "val/images"), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, "val/labels"), exist_ok=True)

# Criação do arquivo de configuração data.yaml
yaml_content = f"""
path: {DATASET_DIR}
train: train/images
val: val/images

names:
  0: placa
"""

yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"✅ data.yaml configurado com sucesso em: {yaml_path}")

## 5. Treinamento do Modelo YOLO
Utilizamos o modelo YOLO com Transfer Learning pré-treinado.

In [ ]:
from ultralytics import YOLO

# Carrega os pesos pré-treinados
modelo = YOLO("yolov8n.pt")

# Inicia o treinamento com GPU
resultados = modelo.train(
    data=os.path.join(DATASET_DIR, "data.yaml"),
    epochs=35,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    name="yolo_placas_modelo"
)

print("🎉 Treinamento finalizado com sucesso!")

## 6. Avaliação das Métricas do Modelo
Visualizamos os gráficos de desempenho e matriz de confusão.

In [ ]:
from IPython.display import Image, display
import glob

for grafico in glob.glob("runs/detect/yolo_placas_modelo/*.png"):
    print(f"📊 Gráfico: {grafico}")
    display(Image(filename=grafico))
    print("-" * 50)

## 7. Download dos Melhores Pesos (`best.pt`)
Baixa o arquivo `best.pt` gerado pelo treinamento para você salvar na pasta `models/best.pt` do seu projeto local.

In [ ]:
from google.colab import files

caminho_pesos = "runs/detect/yolo_placas_modelo/weights/best.pt"

if os.path.exists(caminho_pesos):
    print("⬇️ Baixando arquivo best.pt para o seu computador...")
    files.download(caminho_pesos)
else:
    print("❌ Arquivo best.pt não encontrado. Verifique se o treinamento concluiu com sucesso.")